# Data Processing

Loads all `eval_results.csv` files from a list of top-level experiment folders and merges them into two DataFrames:
- **`sem_var_df`**: one row per semantic variation (aggregated across shuffles), from `eval_results.csv`
- **`replicate_df`**: one row per shuffle/replicate (raw JSONL files)

Run `eval_all.sh` first to generate the `eval_results.csv` files, then run this notebook.

## Setup

In [1]:
import os
os.chdir("../..")

In [2]:
import re
import pandas as pd
import numpy as np
from glob import glob

## Configuration

Add top-level experiment folder paths here. Each entry is a folder containing `model_name_*` subfolders.

In [3]:
toplevel_paths = [
    # Farm domain
    "experiments/20260118_farm_novar",
    "experiments/20260124_farm_4sd_var",
    "experiments/20260124_farm_2sd_var",
    # Abstract Bandit domain
    "experiments/20260120_ab_novar",
    "experiments/20260124_abandit_4sd_var",
    "experiments/20260124_abandit_2sd_var",
    # Clothing Recommendation domain
    "experiments/20260312_rec_novar",
    "experiments/20260312_rec_4sd_var",
    "experiments/20260312_rec_2sd_var",
    # Gemini models
    "experiments/20260311_gemini_farm_novar",
    "experiments/20260312_gemini_ab_novar",
]

## Path Parsing Functions

Auto-detect experimental variables from folder path names:
- **Domain**: from top-level folder name (`farm` → Farm, `ab`/`abandit` → Bandit, `rec` → Clothing Recommendation)
- **Variance**: from top-level folder name (`4sd_var` → Low, `2sd_var` → High, else → No)
- **Model**: from `model_name_[Model]` subfolder name
- **History**: from `prompt_fullhist`/`prompt_summhist` in subfolder name (default: Summarized)

In [4]:
def parse_toplevel_path(toplevel_path):
    """Extract domain and variance from a top-level experiment folder path."""
    name = os.path.basename(toplevel_path.rstrip("/"))

    # Domain
    if "rec" in name:
        domain = "Clothing Recommendation"
    elif "farm" in name:
        domain = "Farm"
    elif "abandit" in name or "ab" in name:
        domain = "Bandit"
    else:
        domain = "Unknown"

    # Variance (4sd = means are far apart = Low noise; 2sd = means closer = High noise)
    if "4sd_var" in name:
        variance = "Low"
    elif "2sd_var" in name:
        variance = "High"
    else:
        variance = "No"

    return domain, variance


def parse_superfolder_name(subfolder_name):
    """Extract model and history from a model_name_* subfolder name."""
    model_match = re.search(r"model_name_(.+?)(?:_prompt_|$)", subfolder_name)
    model = model_match.group(1) if model_match else "Unknown"

    if "prompt_fullhist" in subfolder_name:
        history = "Full"
    elif "prompt_summhist" in subfolder_name:
        history = "Summarized"
    else:
        history = "Summarized"  # default when not specified

    return model, history


def fix_legacy_nomenclature(nomenclature, domain):
    """Map old sem_rel_ nomenclature to the current scheme per domain."""
    if "sem_rel" not in nomenclature:
        return nomenclature
    if domain == "Farm":
        return nomenclature.replace("sem_rel", "world")
    elif domain == "Bandit":
        return nomenclature.replace("sem_rel", "ordinal")
    return nomenclature

## Discover Superfolders

Walk each top-level path and collect all `model_name_*` subfolders with their metadata.

In [5]:
records = []
for toplevel_path in toplevel_paths:
    if not os.path.isdir(toplevel_path):
        print(f"Warning: {toplevel_path} does not exist — skipping.")
        continue
    domain, variance = parse_toplevel_path(toplevel_path)

    for entry in sorted(os.listdir(toplevel_path)):
        if not entry.startswith("model_name_"):
            continue
        superfolder_path = os.path.join(toplevel_path, entry)
        if not os.path.isdir(superfolder_path):
            continue
        model, history = parse_superfolder_name(entry)
        records.append({
            "toplevel_path": toplevel_path,
            "superfolder_path": superfolder_path,
            "domain": domain,
            "model": model,
            "history": history,
            "variance": variance,
        })

superfolders_df = pd.DataFrame(records)
print(f"Found {len(superfolders_df)} superfolders.")
superfolders_df

Found 73 superfolders.


,toplevel_path,superfolder_path,domain,model,history,variance
0,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No
1,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Summarized,No
2,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Olm...,Farm,Olmo-3.1-32B-Instruct,Summarized,No
3,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Qwe...,Farm,Qwen3-14B,Summarized,No
4,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Qwe...,Farm,Qwen3-14B,Full,No
...,...,...,...,...,...,...
68,experiments/20260312_rec_2sd_var,experiments/20260312_rec_2sd_var/model_name_Qw...,Clothing Recommendation,Qwen3-14B,Summarized,High
69,experiments/20260312_rec_2sd_var,experiments/20260312_rec_2sd_var/model_name_Qw...,Clothing Recommendation,Qwen3-32B,Summarized,High
70,experiments/20260312_rec_2sd_var,experiments/20260312_rec_2sd_var/model_name_ge...,Clothing Recommendation,gemini-3.1-flash-lite-preview,Summarized,High
71,experiments/20260311_gemini_farm_novar,experiments/20260311_gemini_farm_novar/model_n...,Farm,gemini-3.1-flash-lite-preview,Summarized,No


## Per-Semantic-Variation DataFrame

Loads `eval_results.csv` from each superfolder (one row per semantic variation, metrics averaged across shuffles).

In [6]:
sem_var_dfs = []
for _, row in superfolders_df.iterrows():
    csv_path = os.path.join(row["superfolder_path"], "eval_results.csv")
    if not os.path.exists(csv_path):
        print(f"No eval_results.csv at {csv_path} — skipping. (Run eval_all.sh first.)")
        continue
    df = pd.read_csv(csv_path)
    df["domain"] = row["domain"]
    df["model"] = row["model"]
    df["history"] = row["history"]
    df["variance"] = row["variance"]
    df["superfolder_path"] = row["superfolder_path"]
    df["nomenclature"] = df["nomenclature"].apply(
        lambda n: fix_legacy_nomenclature(n, row["domain"])
    )
    sem_var_dfs.append(df)

if sem_var_dfs:
    sem_var_df = pd.concat(sem_var_dfs, ignore_index=True)
    print(f"sem_var_df shape: {sem_var_df.shape}")
    sem_var_df.head()
else:
    print("No eval_results.csv files found. Run eval_all.sh first.")
    sem_var_df = pd.DataFrame()

sem_var_df shape: (1505, 169)


In [7]:
sem_var_df[["domain", "model", "history", "variance", "nomenclature", "scale", "n_shuffles"]].value_counts().reset_index(name="count")

,domain,model,history,variance,nomenclature,scale,n_shuffles,count
0,Bandit,Llama3-8B,Summarized,High,alphanumeric,high_neg_scale,10,1
1,Farm,Olmo-3.1-32B-Instruct,Summarized,High,ordinal_helpful,low_scale,10,1
2,Farm,Olmo-3.1-32B-Instruct,Summarized,High,sent_new_mislead,high_neg_scale,10,1
3,Farm,Olmo-3.1-32B-Instruct,Summarized,High,sent_new_helpful,low_scale,10,1
4,Farm,Olmo-3.1-32B-Instruct,Summarized,High,sent_new_helpful,low_neg_scale,10,1
...,...,...,...,...,...,...,...,...
1500,Clothing Recommendation,Llama3-8B,Summarized,Low,ordinal_helpful,high_neg_scale,10,1
1501,Clothing Recommendation,Llama3-8B,Summarized,Low,alphanumeric,low_scale,10,1
1502,Clothing Recommendation,Llama3-8B,Summarized,Low,alphanumeric,low_neg_scale,10,1
1503,Clothing Recommendation,Llama3-8B,Summarized,Low,alphanumeric,high_scale,10,1


In [8]:
sem_var_df.to_csv("experiments/merged_sem_var_results.csv", index=False)
print("Saved experiments/merged_sem_var_results.csv")

Saved experiments/merged_sem_var_results.csv


## Per-Replicate DataFrame

Walks each superfolder's shuffle directories and finds the most recent valid JSONL file (≥10 rows) per shuffle. One row per shuffle/replicate.

In [9]:
# Nomenclatures sorted longest-first to avoid partial substring matches
NOMENCLATURES = sorted([
    "alphanumeric",
    "sem_rel_helpful", "sem_rel_mislead",
    "sent_new_helpful", "sent_new_mislead",
    "sent_helpful", "sent_mislead",
    "ordinal_helpful", "ordinal_mislead",
    "world_helpful", "world_mislead",
], key=len, reverse=True)

SCALES = ["high_neg_scale", "low_neg_scale", "high_scale", "low_scale"]


def find_valid_jsonl(shuffle_path, min_rows=10):
    """Return the most recent JSONL in shuffle_path with at least min_rows lines."""
    candidates = sorted(glob(f"{shuffle_path}/*.jsonl"), key=os.path.getmtime, reverse=True)
    for path in candidates:
        try:
            df = pd.read_json(path, lines=True)
            if len(df) >= min_rows:
                return path
        except Exception:
            continue
    return None


replicate_records = []

for _, sf_row in superfolders_df.iterrows():
    superfolder_path = sf_row["superfolder_path"]

    for sem_var_folder in sorted(os.listdir(superfolder_path)):
        sem_var_path = os.path.join(superfolder_path, sem_var_folder)
        if not os.path.isdir(sem_var_path):
            continue

        shuffle_folder = os.path.join(sem_var_path, "shuffles")
        if not os.path.isdir(shuffle_folder):
            continue

        # Identify nomenclature and scale from folder name
        nom_matches = [n for n in NOMENCLATURES if n in sem_var_folder]
        scale_matches = [s for s in SCALES if s in sem_var_folder]
        if len(nom_matches) != 1 or len(scale_matches) != 1:
            continue

        nomenclature = fix_legacy_nomenclature(nom_matches[0], sf_row["domain"])
        scale = scale_matches[0]

        for shuffle_path in sorted(glob(f"{shuffle_folder}/shuffle_*")):
            jsonl_file = find_valid_jsonl(shuffle_path)
            if jsonl_file is None:
                continue
            shuffle_num = int(os.path.basename(shuffle_path).split("_")[-1])
            replicate_records.append({
                "toplevel_path": sf_row["toplevel_path"],
                "superfolder_path": superfolder_path,
                "domain": sf_row["domain"],
                "model": sf_row["model"],
                "history": sf_row["history"],
                "variance": sf_row["variance"],
                "nomenclature": nomenclature,
                "scale": scale,
                "shuffle_number": shuffle_num,
                "jsonl_file": jsonl_file,
            })

replicate_df = pd.DataFrame(replicate_records)
print(f"replicate_df shape: {replicate_df.shape}")
replicate_df.head()

replicate_df shape: (14768, 10)


,toplevel_path,superfolder_path,domain,model,history,variance,nomenclature,scale,shuffle_number,jsonl_file
0,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No,alphanumeric,high_neg_scale,0,experiments/20260118_farm_novar/model_name_Lla...
1,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No,alphanumeric,high_neg_scale,1,experiments/20260118_farm_novar/model_name_Lla...
2,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No,alphanumeric,high_neg_scale,2,experiments/20260118_farm_novar/model_name_Lla...
3,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No,alphanumeric,high_neg_scale,3,experiments/20260118_farm_novar/model_name_Lla...
4,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No,alphanumeric,high_neg_scale,4,experiments/20260118_farm_novar/model_name_Lla...


In [10]:
replicate_df[["domain", "model", "history", "variance"]].value_counts().reset_index(name="count")

,domain,model,history,variance,count
0,Farm,Qwen3-14B,Summarized,High,360
1,Farm,Qwen3-32B,Summarized,No,360
2,Farm,Qwen3-32B,Summarized,Low,360
3,Farm,Qwen3-32B,Summarized,High,360
4,Farm,Qwen3-14B,Summarized,No,360
5,Farm,Qwen3-14B,Summarized,Low,360
6,Farm,Llama3-8B,Summarized,Low,360
7,Farm,Llama3-8B,Summarized,High,353
8,Clothing Recommendation,Llama3-8B,Summarized,No,290
9,Farm,Olmo-3.1-32B-Instruct,Summarized,High,280


In [11]:
replicate_df.to_csv("experiments/merged_replicate_results.csv", index=False)
print("Saved experiments/merged_replicate_results.csv")

Saved experiments/merged_replicate_results.csv
